# Unit 6 – Practice & Consolidation Worksheet

These exercises are for your own practice and are **not graded**. Work through them after completing the Unit 6 reading material. Attempt each exercise before revealing the answer.

## Exercise types

| Type | What to do |
|---|---|
| Predict the output | Write what you think the code will print *before* running the cell |
| Fix the bug | The code contains an error — find and correct it |
| Fill in the blank | Complete the missing line(s) to produce the expected output |
| Adapt and extend | A working example is given — modify it to solve a related problem |

**Topics:** cross-validation · GridSearchCV and hyperparameter tuning · confusion matrices · precision, recall and F1 · model comparison · final test set evaluation

---

## Section 1 – Cross-Validation

### Exercise 1.1 Predict the output

Without running the cell, predict what will be printed and explain what the `+/-` value represents.

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=300))
])

scores = cross_val_score(pipe, X, y, cv=5)
print(f"Scores: {scores.round(3)}")
print(f"Mean: {scores.mean():.3f} (+/- {scores.std():.3f})")
```

<details>
<summary><strongSelect for answer</strong></summary>

```python
# Scores: 5 accuracy values, each close to 0.96–1.0
# Mean: ~0.973 (+/- ~0.013)
# (exact values depend on the fold splits)
```

`cross_val_score` trains and evaluates the pipeline 5 times, each time using a different fold as the test set and the remaining 4 as training data. The mean is the overall performance estimate; the `+/-` value is the standard deviation across folds — a measure of how consistent the model is. A large standard deviation suggests the estimate is unreliable or the model is sensitive to which data it trains on.

</details>

### Exercise 1.2 Fill in the blank

Complete the code to run 10-fold cross-validation and print the mean and standard deviation of the scores.

```python
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier(n_neighbors=5))
])

scores = cross_val_score(pipe, X, y, cv=________)

print(f"Mean accuracy: {________.round(3)}")
print(f"Std deviation: {________.round(3)}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
scores = cross_val_score(pipe, X, y, cv=10)

print(f'Mean accuracy: {scores.mean().round(3)}')
print(f'Std deviation: {scores.std().round(3)}')
```

`cv=10` creates 10 folds. `scores` is a NumPy array with one accuracy value per fold. `.mean()` gives the overall performance estimate and `.std()` measures variability across folds. More folds give a more stable estimate but take longer to compute — 5 or 10 folds is the standard choice.

</details>

### Exercise 1.3 Fix the bug

The code below has two problems: the scaler is fitted on the full dataset before splitting, and there is no train/test split at all. Fix both issues so that the data is split first and cross-validation is run on training data only, with the scaler inside the pipeline.

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # ← bug 1: no split, scaler sees all data

model = LogisticRegression(max_iter=5000)
scores = cross_val_score(model, X_scaled, y, cv=5)   # ← bug 2: no pipeline
print(scores.mean().round(3))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=5000))
])

scores = cross_val_score(pipe, X_train, y_train, cv=5)
print(scores.mean().round(3))
```

Two fixes are needed. First, split the data before doing anything else — the test set must be held out from all preprocessing and model selection. Second, wrap the scaler and model in a `Pipeline` and pass `X_train`/`y_train` to `cross_val_score`. This ensures the scaler is fitted only on each fold's training portion, not on the validation fold — which would be a subtler form of the same leakage problem.

</details>

### Exercise 1.4 Predict the output

What will the output show, and what does the pattern of scores tell you about the model's stability?

```python
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
import numpy as np

X, y = load_iris(return_X_y=True)

np.random.seed(42)
scores = cross_val_score(
    DecisionTreeClassifier(random_state=42),
    X, y, cv=10
)

print(f"Fold scores: {scores.round(3)}")
print(f"Range: {scores.min():.3f} – {scores.max():.3f}")
print(f"Std: {scores.std():.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Fold scores: 10 values varying between roughly 0.87 and 1.0
# Range: ~0.867 – 1.0
# Std: ~0.04 (higher than logistic regression on same data)
```

Decision trees tend to have higher variance than simpler models — the scores vary more across folds, reflected in a higher standard deviation. A wide range (e.g. 0.87 to 1.0) suggests the model's performance depends heavily on which samples land in the test fold. Comparing standard deviations across models is as important as comparing means when selecting a model.

</details>

### Exercise 1.5 Adapt and extend

The example below runs cross-validation with the default `scoring='accuracy'`:

```python
scores = cross_val_score(pipe, X, y, cv=5)
```

Adapt it to run 5-fold cross-validation **three times** — once with `scoring='accuracy'`, once with `scoring='f1_macro'`, and once with `scoring='roc_auc_ovr'` — on the digits dataset. Print the mean of each.

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=2000))
])

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
for metric in ['accuracy', 'f1_macro', 'roc_auc_ovr']:
    scores = cross_val_score(pipe, X, y, cv=5, scoring=metric)
    print(f'{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})')
```

`cross_val_score` accepts a `scoring` parameter as a string shortcut. `'f1_macro'` computes F1 for each class and averages them, treating all classes equally. `'roc_auc_ovr'` uses a one-vs-rest approach for multi-class AUC. For a balanced dataset like digits, accuracy and F1 macro will be very similar — they diverge most when classes are imbalanced.

</details>

---
## Section 2 – GridSearchCV and Hyperparameter Tuning

### Exercise 2.1 Predict the output

Without running the cell, predict what `grid_search.best_params_` and `grid_search.best_score_` will look like (in terms of structure, not exact values).

```python
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier())
])

param_grid = {
    'model__n_neighbors': [3, 5, 7],
    'model__weights':     ['uniform', 'distance']
}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X, y)

print(grid_search.best_params_)
print(grid_search.best_score_)
print(type(grid_search.best_score_))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# {'model__n_neighbors': <int>, 'model__weights': '<string>'}
# e.g. {'model__n_neighbors': 5, 'model__weights': 'distance'}
# A float between 0 and 1, e.g. 0.973
# <class 'float'>
```

`best_params_` is a dictionary mapping each parameter name to the value that produced the best cross-validated score. The double-underscore `__` notation (`model__n_neighbors`) is how scikit-learn identifies parameters inside a pipeline step. `best_score_` is the mean cross-validated score for the best parameter combination — a single float.

</details>

### Exercise 2.2 Fill in the blank

Complete the `GridSearchCV` setup to tune a `LogisticRegression` inside a pipeline. The grid should search over `C` values `[0.01, 0.1, 1, 10]` and solvers `['lbfgs', 'saga']`, using 5-fold CV.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=5000))
])

param_grid = {
    'model__C':      ________,
    'model__solver': ________
}

grid_search = GridSearchCV(________, ________, cv=________)
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
param_grid = {
    'model__C':      [0.01, 0.1, 1, 10],
    'model__solver': ['lbfgs', 'saga']
}
grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X, y)
```

`C` controls how flexible the model is — a smaller `C` keeps the model simpler, while a larger `C` allows it to fit the training data more closely. The grid has 4 × 2 = 8 combinations; with `cv=5`, this means 40 model fits in total. Passing `pipe` rather than the model directly ensures the scaler is refitted on each training fold during the search, preventing data leakage.

</details>


### Exercise 2.3 Fix the bug

The parameter grid uses the wrong naming convention — it refers to the model directly instead of using the pipeline step name. Fix it.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

pipe = Pipeline([
    ('scaler',     StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

param_grid = {
    'max_depth':        [5, 10, None],    # ← bug
    'min_samples_split': [2, 5, 10]       # ← bug
}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X, y)
print(grid_search.best_params_)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
param_grid = {
    'classifier__max_depth':         [5, 10, None],
    'classifier__min_samples_split': [2, 5, 10]
}
```

When tuning a model inside a `Pipeline`, parameter names must be prefixed with the step name followed by two underscores: `'classifier__max_depth'`, not `'max_depth'`. Without the prefix, `GridSearchCV` raises a `ValueError` because it cannot find `max_depth` as a top-level pipeline parameter.

</details>

### Exercise 2.4 Predict the output

How many model fits will `GridSearchCV` perform in total?

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth':    [10, 20, None],
    'model__max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(pipe, param_grid, cv=5)

n_combinations = len(param_grid['model__n_estimators']) *                  len(param_grid['model__max_depth'])    *                  len(param_grid['model__max_features'])

print(f"Parameter combinations: {n_combinations}")
print(f"Total fits: {n_combinations * 5}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Parameter combinations: 18
# Total fits: 90
# 3 × 3 × 2 = 18 combinations × 5 folds = 90 fits
```

Grid search performs an exhaustive search — every combination is tried. With 3 × 3 × 2 = 18 combinations and 5-fold CV, that is 90 separate model fits. Random forests with many trees are slow to train, so 90 fits can take a significant amount of time. For large grids, `RandomizedSearchCV` samples a random subset of combinations rather than trying all of them.

</details>

### Exercise 2.5 Adapt and extend

The example below retrieves the best score and best parameters after a grid search:

```python
print(grid_search.best_score_)
print(grid_search.best_params_)
```

Adapt it to also:
1. Print the best estimator (the fitted pipeline with best parameters).
2. Extract the top 3 parameter combinations from `cv_results_` and display their mean test scores alongside their parameters.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits
import pandas as pd

X, y = load_digits(return_X_y=True)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  KNeighborsClassifier())
])

param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11],
    'model__weights':     ['uniform', 'distance']
}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X, y)

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
print('Best estimator:', grid_search.best_estimator_)

results = pd.DataFrame(grid_search.cv_results_)
top3 = results.nlargest(3, 'mean_test_score')[['mean_test_score', 'params']]
print('\nTop 3 combinations:')
print(top3.to_string(index=False))
```

`best_estimator_` is the full fitted pipeline retrained on all training data with the best parameters — ready to call `.predict()` on new data. `cv_results_` is a dictionary containing scores and parameters for every combination tried; converting it to a DataFrame and using `.nlargest()` is a clean way to inspect the top-performing configurations.

</details>

---
## Section 3 – Confusion Matrices

### Exercise 3.1 Predict the output

Given this confusion matrix for a binary classifier, what will each print statement output?

```python
import numpy as np
from sklearn.metrics import confusion_matrix

y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
y_pred = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 1])

cm = confusion_matrix(y_true, y_pred)
print(cm)
print(f"TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"FN: {cm[1,0]}, TP: {cm[1,1]}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [[3 1]    ← row 0: actual negative — 3 correct, 1 wrong
#  [2 4]]   ← row 1: actual positive — 2 wrong, 4 correct
# TN: 3, FP: 1
# FN: 2, TP: 4
```

The confusion matrix rows represent the actual class, columns represent the predicted class. `cm[0,0]` (top-left) = True Negatives: correctly predicted as negative. `cm[0,1]` (top-right) = False Positives: wrongly predicted as positive. `cm[1,0]` (bottom-left) = False Negatives: wrongly predicted as negative. `cm[1,1]` (bottom-right) = True Positives: correctly predicted as positive.

</details>

### Exercise 3.2 Fill in the blank

Complete the code to produce and visualise a confusion matrix using a heatmap.

> **Note on labels:** in scikit-learn's breast cancer data the raw encoding is `0 = malignant, 1 = benign`. Here we remap so that **malignant = 1** (the positive class we care about), which also makes the `['Benign', 'Malignant']` axis labels line up correctly with classes `0` and `1`. This matches the "positive = the case to catch" convention used throughout Unit 6.

```python
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
import matplotlib.pyplot as plt
import seaborn as sns

X, y = load_breast_cancer(return_X_y=True)
y = (y == 0).astype(int)        # malignant -> 1 (positive), benign -> 0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
pipe.fit(X_train, y_train)

y_pred = pipe.________(X_test)
cm = confusion_matrix(________, ________)

plt.figure(figsize=(6, 5))
sns.heatmap(________, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix')
plt.tight_layout()
plt.show()
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
y_pred = pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ...)
```

`confusion_matrix(y_true, y_pred)` — the true labels always come first. `annot=True` writes the count inside each cell; `fmt='d'` formats them as integers rather than floats. `cmap='Blues'` gives darker shading to higher values, making the diagonal (correct predictions) visually prominent when the model performs well. Because we remapped so that `1 = malignant`, the axis labels `['Benign', 'Malignant']` (index 0, then index 1) now correctly describe the rows and columns — if you skip the remap, scikit-learn's default `0 = malignant` ordering means those labels are reversed.

</details>


### Exercise 3.3 Fix the bug

The arguments to `confusion_matrix` are in the wrong order. Fix it.

```python
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = np.array([0, 1, 1, 0, 1, 0, 1, 1])
y_pred = np.array([0, 1, 0, 0, 1, 1, 1, 0])

cm = confusion_matrix(y_pred, y_true)    # ← bug
print(cm)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
cm = confusion_matrix(y_true, y_pred)
```

The signature is `confusion_matrix(y_true, y_pred)` — true labels first, predicted labels second. Swapping them transposes the matrix: what appears in the FP cell would actually be the FN count, and vice versa. The diagonal (correct predictions) looks the same either way, which is why this bug can go unnoticed if you only check overall accuracy.

</details>

### Exercise 3.4 Predict the output

Two models have the same accuracy but very different confusion matrices. What will be printed, and what does it reveal about each model?

```python
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

y_true = np.array([0]*90 + [1]*10)   # 90 negatives, 10 positives

# Model A: predicts everything as negative
y_pred_A = np.array([0]*100)

# Model B: gets some positives right but makes more negative errors
y_pred_B = np.array([0]*85 + [1]*5 + [0]*5 + [1]*5)

print("Model A accuracy:", accuracy_score(y_true, y_pred_A))
print("Model B accuracy:", accuracy_score(y_true, y_pred_B))
print()
print("Model A confusion matrix:")
print(confusion_matrix(y_true, y_pred_A))
print()
print("Model B confusion matrix:")
print(confusion_matrix(y_true, y_pred_B))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Model A accuracy: 0.9
# Model B accuracy: 0.9
# 
# Model A:
# [[90  0]
#  [10  0]]   ← never predicts positive; all 10 positives are missed
# 
# Model B:
# [[85  5]
#  [ 5  5]]   ← catches 5 positives but introduces 5 false alarms
```

Both models achieve 90% accuracy on this imbalanced dataset — but Model A is useless: it simply predicts the majority class every time and misses all positives. The confusion matrix exposes this. Model B makes more errors overall but identifies half of the positive cases. In medical screening, missing a positive (false negative) is typically far more costly than a false alarm — accuracy alone would hide this critical difference.

</details>

### Exercise 3.5 Adapt and extend

The example below prints a basic confusion matrix:

```python
cm = confusion_matrix(y_test, y_pred)
print(cm)
```

Adapt it to compute and print **four** derived metrics **manually** from the confusion matrix values, without using any sklearn metric functions:
- Accuracy
- Precision
- Recall (True Positive Rate / Sensitivity)
- Specificity (True Negative Rate)

```python
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
y_pred = np.array([0, 0, 1, 0, 0, 1, 0, 1, 1, 1])

# Your code here
```


<details>
<summary><strong>Select for answer</strong></summary>

```python
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

accuracy    = (TP + TN) / (TP + TN + FP + FN)
precision   = TP / (TP + FP)
recall      = TP / (TP + FN)
specificity = TN / (TN + FP)

print(f'Accuracy:    {accuracy:.3f}')
print(f'Precision:   {precision:.3f}')
print(f'Recall:      {recall:.3f}')
print(f'Specificity: {specificity:.3f}')
```

```python
# Accuracy:    0.800
# Precision:   0.800
# Recall:      0.800
# Specificity: 0.800
```

Accuracy measures overall correctness. Precision asks: of everything predicted positive, how much was right? Recall (sensitivity) asks: of all actual positives, how many were found? Specificity is recall's mirror image for the negative class: of all actual negatives, how many were correctly identified? On this small balanced example all four happen to equal 0.800, but on imbalanced data they typically diverge — which is exactly why a single metric like accuracy can mislead.

</details>


---
## Section 4 – Precision, Recall and F1

### Exercise 4.1 Predict the output

Given the counts below, predict the precision, recall and F1 score before running the cell.

```python
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# TP=40, FP=10, FN=20 (TN not needed for these metrics)
y_true = np.array([1]*60 + [0]*50)
y_pred = np.array([1]*40 + [0]*20 + [1]*10 + [0]*40)

print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Precision: 0.800   (TP=40, FP=10  → 40/(40+10))
# Recall:    0.667   (TP=40, FN=20  → 40/(40+20))
# F1:        0.727   (harmonic mean of 0.8 and 0.667)
```

Precision = TP / (TP + FP): of all predicted positives, how many are correct? Recall = TP / (TP + FN): of all actual positives, how many were found? F1 is the harmonic mean: `2 * (P * R) / (P + R)`. The harmonic mean penalises large differences between precision and recall more than the arithmetic mean would — a model with precision=1.0 and recall=0.0 gets F1=0, not 0.5.

</details>

### Exercise 4.2 Fill in the blank

Complete the code to print a full classification report and identify which class has the lowest F1 score.

```python
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(
    ________,
    ________,
    target_names=[str(i) for i in range(10)]
))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
print(classification_report(
    y_test,
    y_pred,
    target_names=[str(i) for i in range(10)]
))
```

`classification_report` prints precision, recall, F1 and support (number of true instances) for each class, plus macro and weighted averages. `target_names` replaces numeric class labels with readable names. The class with the lowest F1 is typically the one the model struggles with most — often because it looks visually similar to another class (e.g. digits 4 and 9).

</details>

### Exercise 4.3 Fix the bug

The code computes F1 for a multi-class problem but uses the wrong `average` setting, producing a single value that hides per-class differences. The task requires a macro average that treats all classes equally. Fix it.

```python
from sklearn.metrics import f1_score
import numpy as np

y_true = np.array([0, 1, 2, 0, 1, 2, 0, 2, 1, 0])
y_pred = np.array([0, 2, 2, 0, 0, 2, 0, 2, 1, 1])

# Should treat all classes equally regardless of support
score = f1_score(y_true, y_pred, average='micro')    # ← wrong average
print(f"F1: {score:.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
score = f1_score(y_true, y_pred, average='macro')
print(f'F1: {score:.3f}')
```

`average='micro'` pools all TP, FP, FN counts across classes before computing F1 — equivalent to accuracy for balanced datasets and biased towards larger classes. `average='macro'` computes F1 for each class independently and averages the results, giving equal weight to every class regardless of how many samples it has. Use `'macro'` when all classes matter equally; use `'weighted'` when you want class size to influence the average.

</details>

### Exercise 4.4 Predict the output

A spam filter is evaluated below. What will be printed, and which metric matters most for this application?

```python
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# 1 = spam, 0 = not spam
# The filter is very aggressive — it flags almost everything as spam
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 1, 1, 1, 1, 0, 0])

print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
```

In [1]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# 1 = spam, 0 = not spam
# The filter is very aggressive — it flags almost everything as spam
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 1, 1, 1, 1, 0, 0])

print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")


Precision: 0.571
Recall:    0.800


<details>
<summary><strong>Select for answer</strong></summary>

```python
# Precision: 0.500   (4 TP out of 8 predicted positive — many false alarms)
# Recall:    0.800   (4 out of 5 actual spam caught)
```

The filter catches 80% of spam (good recall) but is wrong half the time it flags something (poor precision) — many legitimate emails land in the spam folder. For a spam filter, **precision matters most**: false positives (legitimate emails marked as spam) are very disruptive. A security threat-detection system would have the opposite priority — missing a real threat (low recall) is worse than investigating a false alarm.

</details>


### Exercise 4.5 Adapt and extend

The example below evaluates a model using only accuracy:

```python
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))
```

Adapt it to produce a complete evaluation for a **binary** classifier on the breast cancer dataset, printing:
- Accuracy, precision, recall, F1
- The confusion matrix
- Which metric you would prioritise and why (as a comment)

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'F1:        {f1_score(y_test, y_pred):.3f}')
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

# In cancer detection, recall (sensitivity) is most important:
# a false negative (missed cancer) is more dangerous than a false positive (unnecessary follow-up).
```

For medical diagnosis, the cost of a false negative (missing a real case) almost always outweighs the cost of a false positive (unnecessary investigation). Recall quantifies how many true positives the model finds. A high-recall, lower-precision model is preferable here — you can follow up on false positives, but you cannot undo a missed diagnosis.

</details>

---
## Section 5 – Model Comparison

### Exercise 5.1 Predict the output

Three models are compared. What will the ranking be, and what else should you consider beyond the mean score?

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

models = {
    'Logistic Regression': LogisticRegression(max_iter=300),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5)
}

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=10)
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Logistic Regression: ~0.96 (+/- ~0.05)
# Decision Tree:       ~0.95 (+/- ~0.04)
# KNN:                 ~0.95 (+/- ~0.05)
# (exact values vary; all three are very close)
```

All three models score around 0.95 on Iris, and their means sit **well within one another's standard deviations** — so this is *not* a reliable ranking. Whichever model happens to come top can change from one scikit-learn version to another precisely because the gaps are smaller than the fold-to-fold variation. That is the real lesson: when means are this close, the standard deviation matters as much as the mean, and a difference of one or two hundredths should not be read as one model being genuinely better. Model selection should also weigh interpretability, training time and prediction speed — not just cross-validated accuracy.

</details>


### Exercise 5.2 Fill in the blank

Complete the code to store cross-validation results for three models in a list and identify the best one by mean score.

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

models = [
    ('Logistic Regression', LogisticRegression(max_iter=2000)),
    ('Random Forest',       RandomForestClassifier(n_estimators=50, random_state=42)),
    ('KNN',                 KNeighborsClassifier())
]

results = []

for name, model in models:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=5)
    results.append({'name': name, 'mean': ________, 'std': ________})

best = max(results, key=________)
print(f"Best model: {best['name']} — mean accuracy: {best['mean']:.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
results.append({'name': name, 'mean': scores.mean(), 'std': scores.std()})

best = max(results, key=lambda r: r['mean'])
```

`scores.mean()` and `scores.std()` give the summary statistics for each model. `max(..., key=lambda r: r['mean'])` finds the dictionary entry with the highest mean — the `lambda` extracts the value to compare on. In practice you might also consider `mean - std` as a more conservative selection criterion.

</details>

### Exercise 5.3 Fix the bug

The comparison is unfair because one model is evaluated on a different portion of the data. Fix it so all three models are compared on the same folds.

```python
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

models = {
    'LR':  LogisticRegression(max_iter=5000),
    'DT':  DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier()
}

for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2    # ← bug: no random_state, called inside loop
    )                           # each model gets a different split
    pipe.fit(X_train, y_train)
    print(f"{name}: {pipe.score(X_test, y_test):.3f}")


<details>
<summary><strong>Select for answer</strong></summary>

```python
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f'{name}: {scores.mean():.3f} (+/- {scores.std():.3f})')
```

Using a different random `train_test_split` for each model means each model sees a different test set. A model might score higher simply because it was lucky with which samples ended up in its test set — not because it is genuinely better. Cross-validation with a fixed `cv` value ensures every model is evaluated on exactly the same folds, making the comparison fair.

When `cv=5` is passed to `cross_val_score` with a classifier, scikit-learn automatically uses `StratifiedKFold` — preserving the class distribution in each fold. If you wanted to make this explicit, you could pass the splitter directly:

```python
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# shuffle=True randomises fold assignment before splitting
# random_state=42 ensures the same folds are used every run
scores = cross_val_score(pipe, X, y, cv=cv)
```

This is good practice when you want full control over the splitting behaviour, or when working with a custom estimator that scikit-learn might not recognise as a classifier.

</details>

### Exercise 5.4 Predict the output

What will the printed scores reveal about the trade-off between these two models?

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits
import numpy as np

X, y = load_digits(return_X_y=True)

lr = Pipeline([('scaler', StandardScaler()),
               ('model',  LogisticRegression(max_iter=2000))])

rf = Pipeline([('scaler', StandardScaler()),
               ('model',  RandomForestClassifier(n_estimators=100, random_state=42))])

for name, pipe in [('Logistic Regression', lr), ('Random Forest', rf)]:
    scores = cross_val_score(pipe, X, y, cv=5)
    ci_low  = scores.mean() - 2 * scores.std()
    ci_high = scores.mean() + 2 * scores.std()
    print(f"{name}: {scores.mean():.3f} [{ci_low:.3f} – {ci_high:.3f}]")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Logistic Regression: ~0.921 [~0.884 – ~0.958]
# Random Forest:       ~0.973 [~0.952 – ~0.994]
# (exact values will vary)
```

The `[low – high]` range is an approximate 95% confidence interval (mean ± 2 std). If two confidence intervals overlap substantially, the difference in mean scores may not be meaningful — one model might not genuinely outperform the other. Random Forest typically achieves higher accuracy on the digits dataset but takes longer to train. The confidence interval is a more honest comparison than mean scores alone.

</details>

### Exercise 5.5 Adapt and extend

The example below compares two models by mean CV accuracy:

```python
scores_lr = cross_val_score(lr, X, y, cv=5)
scores_rf = cross_val_score(rf, X, y, cv=5)
print(scores_lr.mean(), scores_rf.mean())
```

Adapt it to compare **four** models on the breast cancer dataset using both `'accuracy'` and `'f1'` scoring, storing results in a dictionary and printing a summary table.

```python
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

models = {
    'LR':  LogisticRegression(max_iter=5000),
    'DT':  DecisionTreeClassifier(random_state=42),
    'RF':  RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier()
}

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
results = {}
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    acc = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    f1  = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    results[name] = {'accuracy': acc.mean(), 'f1': f1.mean()}

print(f"{'Model':<6} {'Accuracy':>10} {'F1':>8}")
print('-' * 26)
for name, scores in results.items():
    print(f"{name:<6} {scores['accuracy']:>10.3f} {scores['f1']:>8.3f}")
```

Running CV twice per model (once per metric) is simple but inefficient — `cross_validate` can return multiple metrics in a single pass. For a medical dataset, the F1 column is arguably more informative than accuracy: it balances precision and recall, which matters when a false negative (missed diagnosis) has serious consequences.

</details>

---
## Section 6 – Final Test Set Evaluation

### Exercise 6.1 Predict the output

What is wrong with the workflow below, even though it produces a number?

```python
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])

param_grid = {'model__C': [0.001, 0.01, 0.1, 1, 10, 100]}

for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    pipe.set_params(model__C=C)
    pipe.fit(X_train, y_train)
    score = pipe.score(X_test, y_test)     # ← problem
    print(f"C={C}: {score:.3f}")

# Pick the C with highest test score and call it the final result
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# The test set is used to select hyperparameters — this is data leakage.
# The chosen C is optimised for the test set, not for generalisation.
# The reported score is overoptimistic.

# Fix: use GridSearchCV with cross-validation on training data only,
# then evaluate the best model ONCE on the test set.

grid_search = GridSearchCV(pipe, {'model__C': [0.001, 0.01, 0.1, 1, 10, 100]}, cv=5)
grid_search.fit(X_train, y_train)
print(f'Best C: {grid_search.best_params_}')
print(f'Final test accuracy: {grid_search.score(X_test, y_test):.3f}')
```

Selecting hyperparameters by evaluating directly on the test set is a subtle but serious form of data leakage. The test set has effectively been used for training decisions, so the reported score is optimistic and will not reflect real-world performance. The test set must be used exactly once — after all model selection decisions are finalised using cross-validation on training data alone.

</details>

### Exercise 6.2 Fill in the blank

Complete the correct workflow: use `GridSearchCV` on training data to select the best model, then evaluate once on the held-out test set.

```python
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [10, None]
}

# Step 1: tune on training data
grid_search = GridSearchCV(________, ________, cv=5)
grid_search.________(X_train, y_train)

print("Best params:", grid_search.________)
print(f"Best CV score: {grid_search.________:.3f}")

# Step 2: evaluate ONCE on test set
final_score = grid_search.________(X_test, y_test)
print(f"Final test accuracy: {final_score:.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print(f'Best CV score: {grid_search.best_score_:.3f}')

final_score = grid_search.score(X_test, y_test)
print(f'Final test accuracy: {final_score:.3f}')
```

After `.fit()`, `grid_search.best_estimator_` is automatically retrained on the full training set using the best parameters — no manual retraining is needed. Calling `.score(X_test, y_test)` on the `GridSearchCV` object delegates to this best estimator. The final test score should be reported as the model's performance estimate; the CV score gives a sense of how reliable that estimate is.

</details>

### Exercise 6.3 Fix the bug

The best model from grid search is re-evaluated on the validation folds rather than the held-out test set. Fix the final evaluation line.

```python
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
param_grid = {'model__C': [0.01, 0.1, 1, 10]}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Final evaluation — should use the held-out test set
final_accuracy = grid_search.best_score_    # ← bug
print(f"Final accuracy: {final_accuracy:.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
final_accuracy = grid_search.score(X_test, y_test)
print(f'Final accuracy: {final_accuracy:.3f}')
```

`best_score_` is the mean cross-validated score on the **training** folds — it is a useful estimate during model selection but is not the final evaluation. The true final evaluation must use `X_test` and `y_test`, which were held out from the entire training and tuning process. These two scores often differ slightly; a large gap suggests overfitting.

</details>

### Exercise 6.4 Predict the output

After the complete workflow below, what will the two accuracy values represent, and why might they differ?

```python
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000))])
param_grid = {'model__C': [0.1, 1, 10]}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"Best CV accuracy:   {grid_search.best_score_:.3f}")
print(f"Test set accuracy:  {grid_search.score(X_test, y_test):.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Best CV accuracy:  ~0.92–0.95  (mean over 5 folds on training data)
# Test set accuracy: ~0.93–0.96  (single evaluation on held-out data)
# Values will be close but rarely identical
```

The CV accuracy is an average over 5 different train/validation splits of the training data — it is the estimate used to select hyperparameters. The test accuracy is a single evaluation on data that was never touched during training or tuning. They should be similar if the model generalises well; a large gap (CV >> test) indicates overfitting, while test >> CV is unusual and may suggest a lucky test split.

</details>

### Exercise 6.5 Adapt and extend

The example below performs the final evaluation and prints accuracy:

```python
final_accuracy = grid_search.score(X_test, y_test)
print(f"Final accuracy: {final_accuracy:.3f}")
```

Adapt it to produce a **complete final evaluation report** after grid search, including:
1. Best parameters found
2. Best CV score
3. Final test accuracy
4. Full classification report on the test set
5. Saving the best pipeline to disk with `joblib`

```python
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.datasets import load_digits
import joblib

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [10, None]
}

grid_search = GridSearchCV(pipe, param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
print('Best parameters:',  grid_search.best_params_)
print(f'Best CV score:     {grid_search.best_score_:.3f}')

y_pred = grid_search.predict(X_test)
print(f'Final test accuracy: {grid_search.score(X_test, y_test):.3f}')

print('\nClassification report:')
print(classification_report(y_test, y_pred,
                            target_names=[str(i) for i in range(10)]))

joblib.dump(grid_search.best_estimator_, 'digits_rf_pipeline.pkl')
print('Pipeline saved to digits_rf_pipeline.pkl')
```

`grid_search.predict(X_test)` delegates to `best_estimator_`, so predictions and scores are always from the best model. Saving `best_estimator_` (the fitted pipeline) rather than the `GridSearchCV` object is cleaner — the saved file contains only what is needed for inference, not the full search history. The classification report shows per-digit performance, revealing which digits the model confuses most often.

</details>

---

## Well done for completing Unit 6 exercises!

If any section felt difficult, revisit the relevant part of the Unit 6 reading material before moving on.

**Key reminders:**

- Cross-validation produces a more reliable performance estimate than a single train-test split — always report the mean and standard deviation across folds
- When using cross-validation, wrap the scaler and model in a `Pipeline` — fitting the scaler on the full dataset before the CV loop is data leakage
- In a `GridSearchCV` parameter grid, use the step name with double underscores to reference pipeline parameters: `'model__C'`, not `'C'`
- `GridSearchCV` performs `n_combinations × n_folds` model fits — large grids with slow models can be very expensive; consider `RandomizedSearchCV` for large search spaces
- The confusion matrix layout is: rows = actual class, columns = predicted class; `cm[0,0]` = TN, `cm[0,1]` = FP, `cm[1,0]` = FN, `cm[1,1]` = TP
- Accuracy is misleading on imbalanced datasets — a model predicting only the majority class can achieve high accuracy while being completely useless
- Precision = TP / (TP + FP); Recall = TP / (TP + FN); F1 is their harmonic mean — choose which to prioritise based on the relative cost of false positives vs false negatives
- Use `average='macro'` in `f1_score` to treat all classes equally; use `average='weighted'` to weight by class frequency
- All hyperparameter selection must happen on training data (via cross-validation) — the test set is used exactly once, for the final reported result
- `grid_search.best_score_` is the CV score used for selection; `grid_search.score(X_test, y_test)` is the true final evaluation — never confuse the two